# 2. 流程增强

## 这一节在解决什么问题

基础 RAG 通常是一次检索、一次生成。6.2 讨论的不是"把 query 写得更像文档"，也不是"把已检索上下文拼得更完整"，而是：当一次请求中的基础流程不够用时，系统如何改变检索、筛选、生成和自评的顺序与条件。

本节的关键词是 **控制流**：什么时候再检索、什么时候先拆问题、什么时候选择不同检索配置、什么时候过滤证据、什么时候让模型检查自己是否应该继续。

## 与第 4 章 / 6.1 / 6.3 的边界

| 章节 | 关注点 | 本节是否展开 |
|---|---|---|
| 第 4 章 Query/Document 对齐 | 让问题更容易命中文档，例如 query rewrite、HyDE、对齐表示 | 不重复实现，只在必要处引用 |
| 6.1 上下文增强 | 检索后如何组织上下文，例如 Sentence Window、Small-to-Big、AutoMerging | 不重复讲上下文拼接细节 |
| 6.2 流程增强 | 单次请求内如何改变 RAG 控制流 | 本节主线 |
| 6.3 系统增强 | 跨请求、多文档、多工具、记忆、agent 编排 | 不前移到本节 |

因此，本节的 query routing 只表示"单次请求内选择检索策略或索引配置"，例如 chunk 粒度、top-k、数学/概念分支；它不是多文档 agent、不是工具调用路由，也不维护跨请求状态。



## 统一实验设置

为了让 6 种方法之间能公平对比，本节固定如下设置（与 6.1 节保持一致）：

| 项 | 值 |
|---|---|
| 知识库 | `../3. 索引阶段/data/pumpkin_book.pdf`（南瓜书《机器学习公式详解》） |
| 评测题集 | `../3. 索引阶段/data/train_dataset.json` 中挑选的 **12 道题**：5 道概念/比较题 + 7 道推导/复杂题（混合题型才能充分暴露各方法的差异） |
| 生成模型 | `glm-4-flash-250414`（智谱 AI） |
| 向量模型 | 本地 `BAAI/bge-small-zh-v1.5` |
| 评估方式 | LLM 作为裁判，0/1/2 三档打分（口径见后文「评估口径」一节） |

**运行前请确保**：

1. 安装依赖：`pip install langchain langchain-community langchain-chroma zhipuai python-dotenv pymupdf pandas modelscope sentence-transformers transformers torch`
2. 在项目根目录的 `.env` 里配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载本地 embedding 模型到 `./models/`

## 加载公共底座

公共工具（embedding、PDF 清洗、Chroma 工厂、LLM 调用、评估循环……）来自 `_common.py`，
不在本节重复展示。本节自己只关心**流程控制**这一层。

> 关于 `llm_call`：本节默认每次成功调用后 `sleep(1)`，是为了避免连续调用 LLM 触发限流『其他 api 可以取消』。

### 准备一个共用的 retriever

6 种方法都基于同一个底层 retriever（`chunk_size=256, chunk_overlap=20, k=4`），
方便后面比较"差异完全来自流程控制"，而不是检索本身。

In [ ]:
import jsonimport reimport sysimport warningsimport pandas as pdfrom pathlib import Pathsys.path.insert(0, ".")from langchain_text_splitters import RecursiveCharacterTextSplitterfrom IPython.display import displayfrom _common import (    get_cleaned_pdf_documents, open_or_build_chroma,    llm_call as _raw_llm_call, build_rag_generation_prompt,    trim_context_to_budget, load_qna_subset,    run_shared_eval, build_compare_table,    CONTEXT_CHAR_BUDGET, QA_PATH,)warnings.filterwarnings("ignore")def llm_call(prompt: str) -> str:    return _raw_llm_call(prompt, sleep_after=1.0)def build_retriever(chunk_size=256, chunk_overlap=20, k=4):    docs = list(get_cleaned_pdf_documents())    splitter = RecursiveCharacterTextSplitter(        chunk_size=chunk_size, chunk_overlap=chunk_overlap,        separators=["\n\n", "\n", " ", ""], keep_separator=True,    )    chunks = splitter.split_documents(docs)    ids = [f"b{i}" for i in range(len(chunks))]    vs = open_or_build_chroma(f"./chroma_db/baseline_{chunk_size}_{chunk_overlap}", chunks, ids)    return vs.as_retriever(search_kwargs={"k": k})retriever = build_retriever()print("✅ retriever 就绪（chunk_size=256, k=4）")

## 本节的统一接口：`*_pipeline(question) -> (answer, trace)`

为了让 6 种方法**互相之间能比较**、也能**复用同一套评估代码**，
本节给所有方法定一条简单的"出入口约定"：

```
def some_pipeline(question: str) -> tuple[str, list[dict]]:
    ...
    return final_answer, trace
```

- **输入**：永远只是 `question` 一个字符串
- **输出**：
  - `final_answer`：给用户的最终回答（这是真正的"产品输出"）
  - `trace`：流程走过的步骤记录（这是"过程输出"，方便我们看清楚发生了什么）

### 为什么不像 6.1 那样只返回字符串？

6.1 节每个方法只负责"拼 context"，所以接口是 `*_context(q) -> str`。
但 6.2 的核心**就是控制流本身**——多轮检索、子问题分解、路由、自反思……
"过程"和"结果"同等重要，所以多返回一个 `trace`，
让我们能在 `inspect_*` 里把它打印出来。

### `trace` 长什么样

每一步是一个 dict：`{"kind": "retrieve", ...}` / `{"kind": "draft", ...}` / `{"kind": "finalize", ...}`，
后面用一个小工具 `print_trace` 顺次打印。


In [ ]:
def step(kind: str, **fields) -> dict:
    """生成一条 trace 记录。kind 常见取值：retrieve / draft / route / grade / reflect / finalize。"""
    return {"kind": kind, **fields}


def print_trace(trace: list[dict]) -> None:
    for i, s in enumerate(trace, start=1):
        kind = s["kind"]
        body = ", ".join(f"{k}={repr(v)[:60]}" for k, v in s.items() if k != "kind")
        print(f"  [{i}] {kind}  {body}")


In [ ]:
def baseline_pipeline(question: str) -> tuple[str, list[dict]]:
    trace: list[dict] = []
    docs = retriever.invoke(question)
    ctx = trim_context_to_budget("\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET)
    trace.append(step("retrieve", branch="baseline", n_hits=len(docs), context_chars=len(ctx)))
    answer = llm_call(build_rag_generation_prompt(question, ctx))
    trace.append(step("finalize", reason="single_pass"))
    return answer, trace


def inspect_baseline(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    answer, trace = baseline_pipeline(question)
    print(f"🧠 最终答案：\n{answer}\n")
    print("🔍 流程 trace：")
    print_trace(trace)


## 评估口径：为什么本节要换打分 prompt

6.1 节用的是"朴素 0/1/2"——LLM 直接看答案像不像、流畅不流畅。
但 6.2 处理的是**多维度复杂题**，比如「为什么要做模型评估？经验误差 vs 泛化误差？哪个更应该最小化？」
这道题里其实藏着 3 个独立角度。

朴素打分的麻烦是：模型答得**流畅但只覆盖了 1 个角度**，照样能拿高分。
所以本节换一个更严格的口径——**维度计分**：

> 让裁判先在心里把参考答案拆成 2~4 个**必答要点**，
> 再看模型答案覆盖了几个，按覆盖比例给 0 / 1 / 2。

接口完全不变（仍是 `simple_eval_2pt(answer, expected, question, prompt_template=...)`），
只是把 `prompt_template` 换成下面的 `DIMENSIONAL_EVAL_PROMPT`。
这样 `build_compare_table` 能继续跟 6.1 / 6.3 的结果拼接。


In [ ]:
DIMENSIONAL_EVAL_PROMPT = (
    "你是判卷人。先在心里把「参考答案」拆成 2~4 个【必答要点】"
    "（如：方法1、方法2、关键条件、关键定义等），不要输出这些要点。\n"
    "然后比对「模型答案」覆盖了几个要点：\n"
    "- 全部覆盖且无关键事实错误：2\n"
    "- 覆盖一半左右、或部分要点表述含糊但方向正确：1\n"
    "- 大部分要点缺失、或关键事实错误：0\n\n"
    "用户问题：{question}\n参考答案：{expected_answer}\n模型答案：{llm_answer}\n\n"
    "仅输出一行，只包含字符 0、1 或 2，不要任何其它文字。"
)


def eval_pipeline(pipeline_fn, qna_dict):
    """6.2 节快捷入口：把 pipeline 返回的 (answer, trace) 适配成 run_shared_eval 需要的 q -> str。"""
    answer_fn = lambda q: pipeline_fn(q)[0]
    return run_shared_eval(answer_fn, qna_dict, eval_prompt_template=DIMENSIONAL_EVAL_PROMPT)


In [ ]:
def flow_method_compare(method_df, method_name):
    return build_compare_table(
        [baseline_df, method_df],
        names=["baseline", method_name],
    )


def flow_method_diff_summary(method_df, method_name):
    df = flow_method_compare(method_df, method_name)
    wins = df.index[df[method_name] > df["baseline"]].tolist()
    regressions = df.index[df[method_name] < df["baseline"]].tolist()
    ties = df.index[df[method_name] == df["baseline"]].tolist()
    return df, wins, regressions, ties


def short_text(text, max_chars=80):
    text = " ".join(str(text).split())
    return text[:max_chars] + ("..." if len(text) > max_chars else "")


def flow_score_summary(compare_df):
    methods = [c for c in compare_df.columns if c != "question"]
    rows = []
    baseline = compare_df["baseline"]
    for method in methods:
        scores = compare_df[method]
        rows.append({
            "method": method,
            "mean_score_0_2": float(scores.mean()),
            "total_score": int(scores.sum()),
            "strict_wins_vs_baseline": 0 if method == "baseline" else int((scores > baseline).sum()),
            "regressions_vs_baseline": 0 if method == "baseline" else int((scores < baseline).sum()),
            "ties_vs_baseline": len(compare_df) if method == "baseline" else int((scores == baseline).sum()),
        })
    return pd.DataFrame(rows)


def trace_has_kind(trace: list[dict], kind: str) -> bool:
    return any(s.get("kind") == kind for s in trace)


def trace_kinds(trace: list[dict]) -> list[str]:
    return [s.get("kind", "") for s in trace]


def validate_demo_trace(method_name: str, trace: list[dict], case_key: str | None = None) -> dict:
    kinds = [s.get("kind", "") for s in trace]
    if method_name == "iterative":
        has_gap_reason = any(s.get("missing") or "缺" in str(s.get("reason", "")) for s in trace)
        reached_max_rounds = any(s.get("reason") == "max_rounds_reached" for s in trace)
        ok = (kinds.count("retrieve") >= 2 or reached_max_rounds) and has_gap_reason
        expected = "至少两次 retrieve，或明确到达 max_rounds；同时必须出现缺口/补检索原因字段"
    elif method_name == "recursive":
        sub_questions = []
        for s in trace:
            if s.get("kind") == "decompose":
                sub_questions.extend(s.get("sub_questions") or s.get("questions") or [])
            if s.get("kind") == "sub_question":
                sub_questions.append(s.get("question") or s.get("sub_question"))
        unique_sub_questions = {str(q).strip() for q in sub_questions if str(q).strip()}
        diversity_review_ok = any(s.get("sub_question_diversity_ok") is True for s in trace)
        sub_answers = [s for s in trace if s.get("kind") == "sub_answer"]
        ok = "decompose" in kinds and len(sub_answers) >= 2 and (len(unique_sub_questions) >= 2 or diversity_review_ok)
        expected = "出现 decompose，至少两个 sub_answer，且 sub_questions 去重后不少于 2 个或 panel 标记 sub_question_diversity_ok"
    elif method_name == "routing":
        routes = [s for s in trace if s.get("kind") == "route"]
        retrieves = [s for s in trace if s.get("kind") == "retrieve"]
        branches = [s.get("branch") for s in retrieves if s.get("branch")]
        expected_branch = {
            "routing_general": "general_index",
            "routing_math": "math_index",
        }.get(case_key)
        has_retrieval_shape = bool(retrieves) and all(
            {"n_hits", "context_chars", "first_hit", "selected_hits"}.issubset(s.keys())
            for s in retrieves
        )
        low_confidence_mixed = bool(routes) and routes[0].get("target") == "mixed_index" and routes[0].get("confidence", 1.0) < 0.55
        branch_ok = expected_branch is None or expected_branch in branches or low_confidence_mixed
        ok = "route" in kinds and has_retrieval_shape and branch_ok
        expected = "出现 route 和对应 branch retrieve；routing_general/routing_math 优先走预期分支，低置信度可走 mixed_index；暴露 branch/n_hits/context_chars/first_hit/selected_hits"
    elif method_name == "adaptive":
        delegates = {s.get("target") or s.get("delegate") for s in trace if s.get("kind") == "delegate"}
        ok = "classify" in kinds and bool(delegates)
        expected = "单题出现 classify 和 delegate；跨 simple/moderate/complex panel 至少覆盖两个 delegate，理想三个"
    elif method_name == "crag":
        tags = [s.get("tag") for s in trace if s.get("kind") == "grade"]
        context_changes = [
            s for s in trace
            if s.get("kind") in {"filter", "rebuild_context"}
            and (
                s.get("before_context_chars") != s.get("after_context_chars")
                or s.get("context_hash_before") != s.get("context_hash_after")
                or (s.get("dropped_count") or 0) > 0
            )
        ]
        ok = any(t in {"partial", "irrelevant"} for t in tags) and bool(context_changes)
        expected = "至少一条证据被判为 partial/irrelevant，且 before/after context chars、dropped_count 或 context hash 证明上下文改变"
    elif method_name == "self_rag":
        verdicts = [s.get("verdict") for s in trace if s.get("kind") == "reflect"]
        has_reflection_verdict = any(v in {"CONTINUE", "FINISH"} for v in verdicts)
        coverage_deltas = [
            s.get("answer_coverage_delta")
            for s in trace
            if isinstance(s.get("answer_coverage_delta"), (int, float))
        ]
        has_non_negative_coverage_delta = any(delta >= 0 for delta in coverage_deltas)
        has_positive_coverage_delta = any(delta > 0 for delta in coverage_deltas)
        has_explainable_stop = any(
            s.get("stop_reason") or s.get("coverage_note") or s.get("failure_location") or s.get("failure_reason")
            for s in trace
        )
        ok = has_reflection_verdict and (has_non_negative_coverage_delta or has_positive_coverage_delta or has_explainable_stop)
        expected = "必须有 reflection verdict；正式通过需要非负/正向覆盖变化或 panel 中明确 baseline 对比收益；失败定位字段只能说明失败，不能替代唯一合格 case"
    else:
        ok = bool(trace)
        expected = "trace 非空"
    return {"method": method_name, "case_key": case_key, "ok": ok, "expected": expected, "kinds": " -> ".join(kinds)}


def eval_pipeline_with_trace(method_name: str, pipeline_fn, qna_dict) -> tuple:
    records_by_question: dict[str, dict] = {}

    def answer_fn(question: str) -> str:
        answer, trace = pipeline_fn(question)
        records_by_question[question] = {
            "method": method_name,
            "question": question,
            "expected": qna_dict[question],
            "answer": answer,
            "trace": trace,
        }
        return answer

    df = run_shared_eval(answer_fn, qna_dict, eval_prompt_template=DIMENSIONAL_EVAL_PROMPT)
    score_by_question = dict(zip(df["question"], df["rag_eval_results"]))

    records = []
    for question, record in records_by_question.items():
        records.append({
            **record,
            "score": score_by_question.get(question),
        })
    return df, records


### 本节题集：流程增强敏感题

6.2 的题集不直接沿用 6.1 的 24 题自然主集，而是选取 12 道更容易暴露“流程差异”的题：概念/比较题负责观察多要点覆盖，推导/复杂题负责观察补检索、拆解、路由和自反思。

注意：本节使用维度计分 prompt，且题集与 6.1 不同，所以分数只用于 6.2 内部横向比较，不与 6.1 的上下文增强分数直接比较。


In [ ]:
# 6.2 流程增强专用主评测集：只用于本节内部比较，不与 6.1 分数直接横比
QA_INDICES_CONCEPT = [0, 1, 4, 5, 7]
QA_INDICES_DERIVE = [12, 15, 18, 19, 36, 42, 86]
QA_INDICES = QA_INDICES_CONCEPT + QA_INDICES_DERIVE
qna_dict = load_qna_subset(QA_PATH, QA_INDICES)
print(f"✅ 本节使用 {len(qna_dict)} 道题（{len(QA_INDICES_CONCEPT)} 概念/比较 + {len(QA_INDICES_DERIVE)} 推导/复杂题）")
print("   本节分数只用于 6.2 内部方法比较；因题集和裁判口径不同，不与 6.1 直接横比。")
print("   想跑得快可只用前 3 道：QA_INDICES = QA_INDICES_CONCEPT[:3]")


Routing 的展示题不是为了“刷平均分”，而是为了暴露它解决的具体失败模式：同一批问题里有些需要大块上下文解释概念，有些需要小块高召回保留公式邻近文本。若题目本身用一个通用检索器已经足够，Routing 可能只增加流程成本；因此 targeted cases 必须能说明 router 为什么改变证据策略。

In [ ]:
DEMO_MINING_CANDIDATES = {
    "iterative": [12, 18, 19, 36, 42, 46, 50, 53, 60, 62, 63, 67, 72, 80, 86, 98, 102, 108, 117],
    "recursive": [4, 7, 42, 62, 63, 86, 101, 113, 117],
    "routing_general": [1, 5, 7, 14, 54, 62, 68, 84, 85, 101, 114, 115],
    "routing_math": [12, 15, 18, 19, 36, 42, 46, 50, 63, 80, 86, 102, 108, 117],
    "adaptive_simple": [14, 22, 33, 35, 84, 85, 107, 114],
    "adaptive_moderate": [4, 5, 7, 17, 38, 54, 62, 68, 101],
    "adaptive_complex": [19, 42, 46, 63, 80, 86, 102, 108, 117],
    "crag": [7, 9, 16, 20, 38, 49, 54, 62, 68, 85, 101, 115],
    "self_rag": [1, 7, 19, 42, 62, 63, 67, 86, 98, 101, 113, 117],
}

DEMO_CASES = {
    "baseline": [0],
    "iterative": [19, 18, 42],
    "recursive": [4, 42, 63],
    "routing_general": [62, 84],
    "routing_math": [12, 19],
    "adaptive_simple": [14, 84],
    "adaptive_moderate": [5, 62],
    "adaptive_complex": [63, 86],
    "crag": [7, 62, 101],
    "self_rag": [1, 62, 63],
}

EXTRA_DEMO_CASES = {
    "iterative_course_natural": "请根据南瓜书相关内容解释牛顿法的基本迭代公式，并说明它与梯度下降法在选择下一个迭代点时的区别。",
    "crag_course_natural": "请根据南瓜书相关内容解释宏平均和微平均的区别，以及它们在类别不平衡时各自可能带来的问题。",
}

_all_demo_indices = sorted({idx for values in DEMO_CASES.values() for idx in values})


def demo_question(case_name: str, pos: int = 0) -> str:
    idx = DEMO_CASES[case_name][pos]
    qna = load_qna_subset(QA_PATH, [idx])
    return next(iter(qna.keys()))


def extra_demo_question(case_name: str) -> str:
    return EXTRA_DEMO_CASES[case_name]


QA_DATA = json.loads(Path(QA_PATH).read_text(encoding="utf-8"))
QA_INDEX_BY_QUESTION = {item["query"]: i for i, item in enumerate(QA_DATA)}


def mine_demo_case(method_name: str, pipeline_fn, candidate_indices: list[int], case_key: str | None = None):
    rows = []
    candidate_qna = load_qna_subset(QA_PATH, candidate_indices)
    for question in candidate_qna.keys():
        answer, trace = pipeline_fn(question)
        validation = validate_demo_trace(method_name, trace, case_key=case_key)
        rows.append({
            "idx": QA_INDEX_BY_QUESTION.get(question),
            "question": question,
            "ok": validation["ok"],
            "case_key": case_key,
            "expected": validation["expected"],
            "trace_shape": validation["kinds"],
            "answer_head": short_text(answer, 140),
        })
    return pd.DataFrame(rows)


print(f"方法展示题已准备：{len(_all_demo_indices)} 道数据集题 + {len(EXTRA_DEMO_CASES)} 道控制题")


## 统一观察框架后面每个方法都回答同一组问题：1. 它改变了 RAG 的哪个流程决策点？2. 它针对什么失败模式？3. 它的 trace 中应该出现什么信号？4. 它相对 baseline 改善了证据、答案覆盖，还是只增加了流程成本？5. 它在哪些题型上可能不值得使用？### 各方法的必须 trace 信号| 方法 | 必须观察的 trace 信号 ||---|---|| Iterative | 是否出现草稿缺口、第二轮 `retrieve` 或明确的停止原因 || Recursive | `decompose` 是否产生互补子问题，`sub_answer` 是否覆盖不同角度 || Routing | `routing_general` / `routing_math` 优先走对应分支；低置信度允许 `mixed_index`；分支的 `branch` / `n_hits` / `context_chars` / `selected_hits` / `first_hit` 至少有可解释差异 || Adaptive | `adaptive_simple` / `adaptive_moderate` / `adaptive_complex` 至少触发两个不同 delegate，理想三类都不同 || CRAG | `grade` 是否把证据标成 `relevant` / `partial` / `irrelevant`，并用 `before_context_chars` / `after_context_chars` / `dropped_count` 或 context hash 证明上下文真的变化 || Self-RAG | `reflect` 后的 `FINISH` / `CONTINUE` 是否带来真实检查效果；至少一个正式 case 需要非负/正向覆盖变化或明确 baseline 对比收益 |

## 六种方法的全景

下面 6 种方法都是在做"流程控制"，但**问的问题不一样**。
我们用一句日常话当导览，再看每种方法在"哪一步"插入决策。

| 方法 | 一句话直觉 | 它在哪一步做决定 |
|---|---|---|
| **迭代检索** | "先答一遍，看缺什么再补一刀" | 生成之后，决定**要不要再检索一轮** |
| **递归检索** | "复杂题先拆成子问题，分头查再合起来" | 检索之前，把**问题本身**拆成几个 |
| **查询路由** | "像医院分诊台，先把病人送对科室" | 检索之前，决定**走哪条索引/工具** |
| **自适应检索** | "看题目难度选战术——简单题一击就走，复杂题排兵布阵" | 检索之前，决定**用多深的策略** |
| **Corrective RAG** | "先验货，挑出靠谱的，再做菜" | 生成之前，给检索结果**打质量分** |
| **Self-RAG** | "边写边改，自问'这答案够不够好'" | 生成中，决定**自己是否要再来一轮** |

### 三类决策点

如果按"决策插在哪里"再归一次类：

```
            ┌─────────────┐    ┌──────────────┐    ┌─────────────┐
question ──▶│  检索 之前   │──▶│   检索 之中   │──▶│  生成 之后   │──▶ answer
            └─────────────┘    └──────────────┘    └─────────────┘
              路由 / 自适应 /        CRAG              迭代 / Self-RAG
              递归（拆题）         （挑证据）            （回头再来）
```

下面我们一个一个看。每种方法都长这样：**直觉 → 流程图 → 代码 → 看 trace**。


## Baseline：单次检索 + 单次生成

在看流程增强之前，先保留一个最朴素的对照组：`retrieve → generate`。后面所有方法都必须和它比较，否则无法判断“多走一步”到底带来了改进，还是只是增加了成本。

Baseline 也使用本节共用的 retriever、同样的 `CONTEXT_CHAR_BUDGET` 和同样的维度计分裁判。


In [ ]:
inspect_baseline(demo_question("baseline"))


## 方法 1：迭代检索（Iterative Retrieval）

> **直觉**：像写考试简答题——先把会的写出来，写到一半发现"哎，KKT 条件没提"，
> 再回去翻书查这一点，然后把答案补全。

### 流程图

```
       ┌─────────────────────────────────────────────────┐
       │                                                  │
       ▼                                                  │
question ─▶ retrieve ─▶ draft（草答 + "还缺什么"）         │
                              │                          │
                              ├─ 缺失=无 ───▶ 输出最终答案  │
                              │                          │
                              └─ 缺失=xxx ──▶ 用 xxx ─────┘
                                              再检索一轮
```

### 适用 / 不适用
- ✅ 适合：单轮答得到一部分、但有明显遗漏的题（多角度题尤其常见）
- ❌ 不适合：信息根本不在知识库里——再补几轮也补不出
- ⚠️ 成本：每多一轮就多 1 次 retrieve + 1 次 LLM 调用


In [ ]:
ITERATIVE_DRAFT_PROMPT = (
    "问题：{question}\n上下文：\n{context}\n\n"
    "请先给出当前答案，再判断信息是否已经足够。严格按下面两行格式输出：\n"
    "当前答案：...\n"
    "状态：[COMPLETE] 或 [NEED_MORE] xxx（xxx 用一句话说明还缺哪类信息）"
)


def _extract_answer(draft: str) -> str:
    """从 draft 中提取「当前答案：」后面的正文，剥掉「状态：」行和标记。"""
    body = draft
    if "状态：" in body:
        body = body.split("状态：", 1)[0]
    if "当前答案：" in body:
        body = body.split("当前答案：", 1)[1]
    return body.strip()


def iterative_pipeline(question: str, max_rounds: int = 2) -> tuple[str, list[dict]]:
    trace: list[dict] = []
    context_blocks: list[str] = []
    missing_hint = ""
    merged_ctx = ""

    for i in range(max_rounds):
        search_q = question if not missing_hint else f"{question}\n补充线索：{missing_hint}"
        docs = retriever.invoke(search_q)
        context_blocks.extend(d.page_content for d in docs)
        merged_ctx = trim_context_to_budget("\n\n".join(context_blocks[-8:]), CONTEXT_CHAR_BUDGET)
        trace.append(step("retrieve", round=i + 1, hint=missing_hint, n_hits=len(docs)))

        draft = llm_call(ITERATIVE_DRAFT_PROMPT.format(question=question, context=merged_ctx))
        trace.append(step("draft", round=i + 1, head=draft[:80]))

        if "[COMPLETE]" in draft:
            trace.append(step("finalize", reason="self_complete"))
            return _extract_answer(draft), trace

        if "[NEED_MORE]" in draft:
            missing_hint = draft.split("[NEED_MORE]", 1)[1].strip()[:120]
        else:
            missing_hint = ""

    final = llm_call(build_rag_generation_prompt(question, merged_ctx))
    trace.append(step("finalize", reason="max_rounds_reached"))
    return final, trace


def inspect_iterative(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    answer, trace = iterative_pipeline(question)
    print(f"🧠 最终答案：\n{answer}\n")
    print("🔍 流程 trace：")
    print_trace(trace)


inspect_iterative(demo_question("iterative"))


### 看 trace 学到了什么

观察上面的 trace，你应该看到典型的两轮：

```
[1] retrieve  round=1, hint='', n_hits=4         ← 第一轮：用原问题检索
[2] draft     round=1, head='当前答案：…  状态：[NEED_MORE] 还需要 KKT 条件…'
[3] retrieve  round=2, hint='还需要 KKT 条件…',   ← 第二轮：把"缺啥"作为补充线索再检索
              n_hits=4
[4] draft     round=2, head='当前答案：…  状态：[COMPLETE]'
[5] finalize  reason='self_complete'
```

**关键观察**：第 2 步的 `[NEED_MORE]` 线索成了第 3 步检索的补充查询——
这就是"流程"的精髓：**让中间结果驱动下一步的输入**。

### 局限：缺失的子问题是"猜"出来的

迭代检索完全靠 LLM 自己说"还缺什么"。但 LLM 自己可能就**意识不到自己缺啥**——
你不知道你不知道的事。如果问题本身能**显式拆**成独立的子问题
（"对偶优势是什么 / KKT 条件的作用 / 强对偶性条件"），
直接拆比让模型猜更可控。这就是下一种方法。
本次运行中代表题一轮即完成，说明教学版的 `[NEED_MORE]` 触发依赖 LLM 自评。若要观察稳定的第二轮，可尝试把代表题切换到 Hessian、SVM 核函数或 EM/GMM 参数估计这类长推导题。


## 方法 2：递归分解（Recursive Decomposition）

> **直觉**：像组队完成大作业——组长把"研究 SVM"拆成
> "原理是什么 / 怎么解 / 有什么坑"，
> 每个组员独立查资料、写一段，最后组长把三段拼成完整报告。

### 流程图

```
question ─▶ decompose ─▶ sub_q1 ─▶ retrieve ─▶ generate ─▶ sub_ans1 ─┐
                     ├─▶ sub_q2 ─▶ retrieve ─▶ generate ─▶ sub_ans2 ─┼─▶ merge ─▶ answer
                     └─▶ sub_q3 ─▶ retrieve ─▶ generate ─▶ sub_ans3 ─┘
```

### 与迭代检索的关键区别

|  | 迭代检索 | 递归分解 |
|---|---|---|
| 决策时机 | 先生成草稿，再根据缺口决定是否补检索 | 生成前先拆问题，再分别检索子问题 |
| 解决的问题 | 首轮答案漏要点、证据不足、需要补查 | 原问题包含多个子任务或多跳依赖 |
| trace 信号 | 多次 `retrieve`，或出现缺口提示与继续检索理由 | `decompose` 后出现多个 `sub_answer` |
| 主要风险 | LLM 不知道自己缺什么，可能过早停止 | 子问题拆错或重复，可能稀释主问题 |

### 适用 / 不适用
- ✅ 适合：能被人类一眼看出"这其实是 3 件事"的复合题
- ❌ 不适合：本质上一气呵成的题（比如"小明的爸爸是谁？"）
- ⚠️ 上限取决于"拆得对不对"——拆错就全错




In [ ]:
DECOMPOSE_PROMPT = (
    "任务：将以下复杂问题拆成 2~3 个独立的、可单独检索的子问题。\n"
    "主问题：{question}\n"
    "要求：每行一个子问题，不要序号、不要解释。"
)

MERGE_PROMPT = (
    "主问题：{question}\n\n"
    "已知子问题与子答案：\n{sub_block}\n\n"
    "请基于上述子答案整合一个结构化的最终回答。"
)


def recursive_pipeline(question: str, sub_questions: list[str] | None = None) -> tuple[str, list[dict]]:
    trace: list[dict] = []

    if sub_questions is None:
        raw = llm_call(DECOMPOSE_PROMPT.format(question=question))
        sub_questions = [q.strip() for q in raw.splitlines() if q.strip()] or [question]

    trace.append(step("decompose", n_subs=len(sub_questions),
                      subs=[q[:40] for q in sub_questions]))

    sub_answers = []
    for i, sq in enumerate(sub_questions, start=1):
        docs = retriever.invoke(sq)
        ctx = trim_context_to_budget("\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET)
        ans = llm_call(build_rag_generation_prompt(sq, ctx))
        sub_answers.append((sq, ans))
        trace.append(step("sub_answer", idx=i, sub_q=sq[:60], ans_head=ans[:60]))

    sub_block = "\n\n".join(f"[子问题{i}] {q}\n[子答案{i}] {a}"
                           for i, (q, a) in enumerate(sub_answers, start=1))
    final = llm_call(MERGE_PROMPT.format(question=question, sub_block=sub_block))
    trace.append(step("finalize", reason="merged_from_subs"))
    return final, trace


def inspect_recursive(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    answer, trace = recursive_pipeline(question)
    print(f"🧠 最终答案：\n{answer}\n")
    print("🔍 流程 trace：")
    print_trace(trace)


inspect_recursive(demo_question("recursive"))


### 看 trace 学到了什么

trace 应当大致是：

```
[1] decompose   n_subs=3, subs=['为什么要做模型评估', '经验误差的定义', '泛化误差的定义']
[2] sub_answer  idx=1, sub_q='为什么要做模型评估', ans_head='...'
[3] sub_answer  idx=2, sub_q='经验误差的定义',     ans_head='...'
[4] sub_answer  idx=3, sub_q='泛化误差的定义',     ans_head='...'
[5] finalize    reason='merged_from_subs'
```

**关键观察**：每个子问题都有独立的 retrieve→generate，
最后一次 merge 把这些子答案缝合起来。

### 一个有意思的对比

迭代和递归都在解决"多走几步"，但还没回答一个更前置的问题：**走哪条路？**
当系统手里有多个索引（数学公式索引 / 概念解释索引 / 代码索引……）或多个工具时，
**选错路**意味着无论走多少步都找不到答案。这就是查询路由。

## 方法 3：查询路由（Query Routing）

> **直觉**：像医院的分诊台——病人走进来先描述症状，分诊台决定送去**内科还是外科**，
> 而不是把所有人都塞进同一个全科门诊。

### 流程图

```
                    ┌─▶ math_index   ─▶ retrieve ─┐
question ─▶ router ─┤                              ├─▶ generate ─▶ answer
                    └─▶ general_index ─▶ retrieve ─┘
```

router 通常有两种实现：
- **规则路由**：用关键词匹配（如出现"公式"→ math_index）。简单稳定，但维护规则烦人
- **LLM 路由**：让 LLM 自己判断走哪条。灵活，但多一次模型调用且会出错

> **本节的 routing 边界**：这里的 query routing 只发生在单次请求内部，用来选择检索策略或索引配置。教学实现使用同一本南瓜书构造两个不同 branch：
>
> - `general_index`：较大 chunk、较少 top-k，适合概念解释和比较题；
> - `math_index`：较小 chunk、更多 top-k，适合公式、推导、矩阵、优化类题。
>
> 它不负责多文档 agent、工具调用路由、跨请求记忆或长期状态。那些属于 6.3 系统增强。

### 适用 / 不适用
- ✅ 适合：知识库**真的拆成了多个索引**（不同领域、不同粒度、不同来源）
- ❌ 不适合：只有一个索引——路由没意义
- ⚠️ 路由错=后面全错，所以路由本身的准确率很关键

> **本节的教学实现**：为了让路由有真实差异，我们仍然使用同一本南瓜书，但构造两个不同检索配置：
> - `general_index`：较大 chunk、较少 top-k，适合概念解释题；
> - `math_index`：较小 chunk、更多 top-k，适合公式、推导、矩阵类题。
>
> 这仍然不是生产中的完整多源路由；真实系统里可以把两个 branch 换成不同知识库、关键词索引、SQL 或工具调用。



In [ ]:
ROUTER_PROMPT = """你是 RAG 查询路由器。请根据问题选择最合适的检索策略。可选策略：- math_index：题目要求推导、写公式、解释数学表达式、KKT/梯度/Hessian/核函数/EM/MM 等数学步骤。- general_index：题目要求定义、概念区别、优缺点、应用背景，不需要展开公式推导。- mixed_index：题目同时包含概念解释和公式推导，或你不确定单一索引是否足够。只输出 JSON，不要 Markdown，不要解释：{{"target": "math_index|general_index|mixed_index", "confidence": 0.0到1.0之间的小数, "reason": "不超过20个字"}}问题：{question}""".strip()def parse_router_decision(raw: str) -> dict:    """Parse router JSON and fall back to mixed_index on malformed or low-confidence output."""    text = str(raw).strip()    try:        if "```" in text:            text = text.split("```")[1]            text = text.removeprefix("json").strip()        data = json.loads(text)    except Exception:        lowered = text.lower()        if "math" in lowered and "general" not in lowered:            data = {"target": "math_index", "confidence": 0.7, "reason": "legacy math"}        elif "general" in lowered and "math" not in lowered:            data = {"target": "general_index", "confidence": 0.7, "reason": "legacy general"}        else:            data = {"target": "mixed_index", "confidence": 0.0, "reason": "parse failed"}    target = str(data.get("target", "mixed_index")).strip().lower()    if target not in {"math_index", "general_index", "mixed_index"}:        target = "mixed_index"    try:        confidence = float(data.get("confidence", 0.0))    except Exception:        confidence = 0.0    confidence = max(0.0, min(1.0, confidence))    if confidence < 0.55:        target = "mixed_index"    return {        "target": target,        "confidence": confidence,        "reason": str(data.get("reason", ""))[:40],    }assert parse_router_decision('{"target":"math_index","confidence":0.9,"reason":"公式推导"}')["target"] == "math_index"assert parse_router_decision('{"target":"general_index","confidence":0.4,"reason":"不确定"}')["target"] == "mixed_index"assert parse_router_decision('math_index')["target"] == "math_index"assert parse_router_decision('nonsense')["target"] == "mixed_index"def select_routing_evidence(question: str, docs: list, max_docs: int = 4) -> list:    """Keep evidence that overlaps with the question while preserving retriever order."""    question_tokens = {        token.lower()        for token in re.findall(r"[A-Za-z0-9_]+|[一-鿿]", question)        if token.strip()    }    scored = []    seen = set()    for rank, doc in enumerate(docs):        content = doc.page_content        fingerprint = content[:120]        if fingerprint in seen:            continue        seen.add(fingerprint)        doc_tokens = {            token.lower()            for token in re.findall(r"[A-Za-z0-9_]+|[一-鿿]", content[:1200])            if token.strip()        }        overlap = len(question_tokens & doc_tokens)        formula_bonus = 2 if any(symbol in content for symbol in ["=", "∑", "∂", "∇", "β", "λ", "argmin"]) else 0        scored.append((overlap + formula_bonus, -rank, doc))    scored.sort(reverse=True, key=lambda item: (item[0], item[1]))    selected = [doc for _, _, doc in scored[:max_docs]]    return selected or docs[:max_docs]general_retriever = build_retriever(chunk_size=512, chunk_overlap=60, k=2)math_retriever = build_retriever(chunk_size=128, chunk_overlap=20, k=8)INDEX_TO_RETRIEVER = {    "math_index": math_retriever,    "general_index": general_retriever,}def mixed_retrieve(question: str) -> tuple[list, dict]:    math_docs = math_retriever.invoke(question)    general_docs = general_retriever.invoke(question)    selected = select_routing_evidence(question, math_docs[:4] + general_docs[:3], max_docs=4)    diagnostics = {        "math_hits": len(math_docs),        "general_hits": len(general_docs),        "selected_hits": len(selected),    }    return selected, diagnosticsdef routing_pipeline(question: str) -> tuple[str, list[dict]]:    trace: list[dict] = []    raw_decision = llm_call(ROUTER_PROMPT.format(question=question))    decision = parse_router_decision(raw_decision)    branch = decision["target"]    trace.append(step(        "route",        target=branch,        confidence=decision["confidence"],        reason=decision["reason"],        raw=raw_decision[:120],    ))    if branch == "mixed_index":        docs, diagnostics = mixed_retrieve(question)    else:        chosen_retriever = INDEX_TO_RETRIEVER[branch]        docs = chosen_retriever.invoke(question)        docs = select_routing_evidence(question, docs, max_docs=min(4, len(docs)))        diagnostics = {"selected_hits": len(docs)}    ctx = trim_context_to_budget("\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET)    trace.append(step(        "retrieve",        branch=branch,        n_hits=len(docs),        context_chars=len(ctx),        first_hit=docs[0].page_content[:80] if docs else "",        **diagnostics,    ))    answer = llm_call(build_rag_generation_prompt(question, ctx))    trace.append(step("finalize", reason="answered"))    return answer, tracedef inspect_routing(question: str) -> None:    print(f"❓ 问题: {question}\n")    answer, trace = routing_pipeline(question)    print(f"🧠 最终答案：\n{answer}\n")    print("🔍 流程 trace：")    print_trace(trace)inspect_routing(demo_question("routing_general"))inspect_routing(demo_question("routing_math"))

### 看 trace 学到了什么

```
[1] route      target='general_index'
[2] retrieve   branch='general_index', n_hits=4
[3] finalize   reason='answered'
```

只比 baseline 多了一步 `route`——但这一步的价值在生产环境会被放大：
当索引数量从 1 涨到 5、10、100，路由的好坏直接决定整个系统的命中率。

### 接下来：走"哪条路"之外，还要决定走"多深"

路由解决的是**横向选择**（走哪条索引），但还有一个**纵向选择**没回答：
- 简单事实查询，一次检索就够
- 多维度复杂题，要迭代或递归

让系统**自动**根据问题复杂度选择检索深度，就是下一种方法：自适应检索。
如果 router 在某次运行中选错 branch，不要把它当成 notebook 错误；这正是查询路由的核心风险：路由层一旦误判，后面的检索和生成都会被带偏。


## 方法 4：自适应检索（Adaptive Retrieval）

> **直觉**：像老司机做菜——
> "炒个青菜直接下锅"，"红烧肉得焯水煸炒慢炖"。
> 同样是做菜，**根据难度选战术**，而不是所有菜都一套流程走到底。

### 流程图

```
question ─▶ classify（complexity?）
                ├─ simple    ─▶ 单次检索 + 生成
                ├─ moderate  ─▶ iterative_pipeline（先答后补）
                └─ complex   ─▶ recursive_pipeline（拆题分工）
```

注意自适应**自己不做检索**——它是一个**调度器**，把活分给前面三种方法之一。

### 与查询路由的对比

|  | 查询路由 | 自适应检索 |
|---|---|---|
| 决定的是 | 走**哪条**索引（横向） | 走**多深**的策略（纵向） |
| 类比 | 分诊到内科 / 外科 | 选门诊 / 住院 / ICU |

两者完全可以**叠加**：先路由选索引，再自适应选深度。
（本节为了解耦演示，把它们拆开看。）

### 适用 / 不适用
- ✅ 适合：用户提问难度跨度大（既有事实查询又有研究型问题）
- ❌ 不适合：所有问题难度都差不多——多套这一层是浪费

In [ ]:
CLASSIFIER_PROMPT = (
    "请判断下列问题的复杂度，仅输出: simple / moderate / complex\n"
    "- simple：事实查询，单一知识点\n"
    "- moderate：需要对比或补齐信息\n"
    "- complex：多维度复杂问题，需拆解\n"
    "问题：{question}"
)


def _classify_complexity(question: str) -> str:
    raw = llm_call(CLASSIFIER_PROMPT.format(question=question)).strip().lower()
    if "complex" in raw:
        return "complex"
    if "moderate" in raw:
        return "moderate"
    return "simple"


def adaptive_pipeline(question: str) -> tuple[str, list[dict]]:
    trace: list[dict] = []

    complexity = _classify_complexity(question)
    trace.append(step("classify", complexity=complexity))

    if complexity == "complex":
        answer, sub_trace = recursive_pipeline(question)
        trace.append(step("delegate", to="recursive_pipeline"))
        trace.extend(step("  └ " + s["kind"], **{k: v for k, v in s.items() if k != "kind"})
                     for s in sub_trace)
    elif complexity == "moderate":
        answer, sub_trace = iterative_pipeline(question)
        trace.append(step("delegate", to="iterative_pipeline"))
        trace.extend(step("  └ " + s["kind"], **{k: v for k, v in s.items() if k != "kind"})
                     for s in sub_trace)
    else:
        docs = retriever.invoke(question)
        ctx = trim_context_to_budget("\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET)
        answer = llm_call(build_rag_generation_prompt(question, ctx))
        trace.append(step("delegate", to="single_pass"))

    trace.append(step("finalize", reason=f"via_{complexity}"))
    return answer, trace


def inspect_adaptive(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    answer, trace = adaptive_pipeline(question)
    print(f"🧠 最终答案：\n{answer}\n")
    print("🔍 流程 trace（含子 pipeline 的步骤）：")
    print_trace(trace)


inspect_adaptive(demo_question("adaptive_simple"))
inspect_adaptive(demo_question("adaptive_moderate"))
inspect_adaptive(demo_question("adaptive_complex"))


### 看 trace 学到了什么

```
[1] classify   complexity='complex'
[2] delegate   to='recursive_pipeline'
[3]   └ decompose   n_subs=3, ...
[4]   └ sub_answer  idx=1, ...
[5]   └ sub_answer  idx=2, ...
[6]   └ sub_answer  idx=3, ...
[7]   └ finalize    reason='merged_from_subs'
[8] finalize   reason='via_complex'
```

trace 把**自适应自己的步骤**和**子 pipeline 的步骤**都展开了——
本质上 adaptive 只是在最外层多套了一个 `classify + delegate`。

### 一个还没被解决的问题

到这里我们解决了"多走几步"（迭代/递归）、"走哪条路"（路由）、"走多深"（自适应）。
但**所有方法都默认了一件事：检索回来的内容是可信的**。

如果 retriever 一次返回 4 条，其中 3 条根本不沾边呢？
后面流程再精巧，也是在垃圾上盖楼。下一种方法专门解决这个。

## 方法 5：Corrective RAG（CRAG）

> **直觉**：像超市验货员——快递送来一筐水果，
> 先**逐个挑出烂的扔掉**再上架，而不是整筐直接卖。

### 流程图

```
question ─▶ retrieve ─▶ 对每条证据打标 ─▶ 走对应分支 ─▶ generate ─▶ answer

  打标结果              用哪些证据                          分支
  ─────                ─────────                          ────
  全部 relevant         所有证据都用                        all_relevant
  部分 relevant/partial 只用 relevant + partial（最多 4 条） partially_relevant
  全部 irrelevant       提示"证据不足，建议重写问题"         irrelevant
```

### 关键洞察

CRAG 的判分由**另一次 LLM 调用**完成（叫 grader），
而不是依赖 retriever 自己的 score。这样做的代价是：
- 每检索 4 条 → 多 4 次 LLM 调用
- 但换来的是**对噪声的免疫力**

### 适用 / 不适用
- ✅ 适合：知识库噪声多、或者用户的问题很容易触发"擦边检索"
- ❌ 不适合：知识库本身很干净——多花一倍 LLM 调用没收益
- ⚠️ grader 本身可能误判，把对的扔了——所以分级用 relevant/partial/irrelevant 三档而不是 yes/no


In [ ]:
GRADER_PROMPT = (
    "判断下面文档片段和问题的相关性，只输出 relevant / partial / irrelevant 之一。\n"
    "问题：{question}\n片段：{snippet}"
)


def grade_relevance(question: str, text: str) -> str:
    raw = llm_call(GRADER_PROMPT.format(question=question, snippet=text[:600])).lower()
    if "irrelevant" in raw:
        return "irrelevant"
    if "partial" in raw:
        return "partial"
    return "relevant"


def crag_pipeline(question: str) -> tuple[str, list[dict]]:
    trace: list[dict] = []
    docs = retriever.invoke(question)

    bucket: dict[str, list[str]] = {"relevant": [], "partial": [], "irrelevant": []}
    for i, d in enumerate(docs, start=1):
        tag = grade_relevance(question, d.page_content)
        bucket[tag].append(d.page_content)
        trace.append(step("grade", idx=i, tag=tag))

    if len(bucket["relevant"]) == len(docs):
        pieces, branch = bucket["relevant"], "all_relevant"
    elif bucket["relevant"] or bucket["partial"]:
        pieces, branch = (bucket["relevant"] + bucket["partial"])[:4], "partially_relevant"
    else:
        pieces, branch = ["检索证据不足，请先重写问题后再检索。"], "irrelevant"
    trace.append(step("route", branch=branch, n_used=len(pieces)))

    before_ctx = "\n\n".join(d.page_content for d in docs)
    dropped_count = len(bucket["irrelevant"])
    ctx = trim_context_to_budget("\n\n".join(pieces), CONTEXT_CHAR_BUDGET)
    trace.append(step("rebuild_context", before_context_chars=len(before_ctx), after_context_chars=len(ctx), dropped_count=dropped_count))
    answer = llm_call(build_rag_generation_prompt(question, ctx))
    trace.append(step("finalize", reason=branch))
    return answer, trace


def inspect_crag(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    answer, trace = crag_pipeline(question)
    print(f"🧠 最终答案：\n{answer}\n")
    print("🔍 流程 trace：")
    print_trace(trace)


inspect_crag(demo_question("crag"))
inspect_crag(extra_demo_question("crag_course_natural"))




### 看 trace 学到了什么

```
[1] grade     idx=1, tag='relevant'
[2] grade     idx=2, tag='partial'
[3] grade     idx=3, tag='irrelevant'      ← 被 CRAG 挡掉
[4] grade     idx=4, tag='relevant'
[5] route     branch='partially_relevant', n_used=3
[6] finalize  reason='partially_relevant'
```

可以直观看到：原本 4 条都会被喂给 LLM，**CRAG 把 idx=3 那条挡掉了**。
最终送进 LLM 的上下文里没有那条噪声。

### 三种分支的现实意义

| 分支 | 出现频率 | 该怎么处理 |
|---|---|---|
| `all_relevant` | 不太常见（理想情况） | 直接生成 |
| `partially_relevant` | **最常见** | 过滤后生成 |
| `irrelevant` | 偶发（问题超出知识范围） | 提示用户重写、或触发补检索 |

### CRAG 的局限：只看一次

CRAG 的决策是**一次性的**——评估一次、处理一次就结束了。
如果我们希望系统能**持续自问"这答案够好吗？要不要再来一轮？"**，
就需要把"反思"做成循环。这就是 Self-RAG 的思想。

## 方法 6：Self-RAG（教学化简版）

> **直觉**：像作者**自己审稿**——
> 写完一段先停下来想"这写得够清楚吗？还需要再查点资料吗？"
> 觉得 OK 就交稿，觉得不行就回去补。

### 流程图

```
       ┌────────────────────────────────────────────┐
       ▼                                            │
question ─▶ retrieve ─▶ generate-and-critique       │
                              │                     │
                              ├─ [FINISH]   ─▶ 输出  │
                              │                     │
                              └─ [CONTINUE] ────────┘
                                  （自评不足，再补检索）
```

注意它和**迭代检索**很像，但视角不一样：

|  | 迭代检索 | Self-RAG |
|---|---|---|
| 反思的是 | "**信息**够不够" | "**回答**够不够" |
| 何时停 | 信息覆盖 → 停 | 回答自评满意 → 停 |
| 出处 | 工程经验 | 论文方法（Asai 等，2023） |

### 教学版 vs 论文原版

| | 教学版（本节） | 论文原版 |
|---|---|---|
| 怎么决策"继续/停止" | 让通用 LLM 输出 `[FINISH]` / `[CONTINUE]` 标记 | **专门微调**模型，让它输出特殊 token：`[Retrieve]` / `[IsRel]` / `[IsSup]` |
| 模型 | 任何 LLM API | `selfrag_llama2_7b` 等专门模型 |
| 决策精度 | 依赖通用 LLM 的判断力 | 内置反思 token，更稳 |
| 成本 | 一次 LLM 调用 | 需要本地部署专门模型 |

教学版保留了**控制逻辑**，但牺牲了原版的"内置反思 token"精度。
真正上生产，建议看论文原版的 reflection token 方案。

### 适用 / 不适用
- ✅ 适合：质量优先于成本、需要"边写边反思"的场景
- ❌ 不适合：低延迟需求——`[CONTINUE]` 一次就翻倍延迟


In [ ]:
SELF_RAG_PROMPT = (
    "问题：{question}\n上下文：{context}\n当前草稿：{draft}\n\n"
    "请按下面规则之一作答：\n"
    "1. 若上下文已足够完整准确地回答，给出最终答案，并以 [FINISH] 结尾。\n"
    "2. 若仍不够，给出当前已知片段，并以 [CONTINUE] xxx 结尾（xxx 说明还需检索什么）。"
)


def _strip_self_rag_markers(response: str) -> str:
    """剥掉 [FINISH] / [CONTINUE] xxx 这类 meta 标记，只留正文给评估。"""
    body = response.split("[CONTINUE]", 1)[0]
    return body.replace("[FINISH]", "").strip()


def self_rag_pipeline(question: str, max_steps: int = 2) -> tuple[str, list[dict]]:
    trace: list[dict] = []
    context_list: list[str] = []
    draft = ""

    for i in range(max_steps):
        search_q = question if not draft else f"{question} (补充检索: {draft[:50]}...)"
        docs = retriever.invoke(search_q)
        context_list.extend(d.page_content for d in docs)
        ctx = trim_context_to_budget("\n\n".join(context_list[-4:]), CONTEXT_CHAR_BUDGET)
        trace.append(step("retrieve", step_no=i + 1, n_hits=len(docs)))

        response = llm_call(SELF_RAG_PROMPT.format(question=question, context=ctx, draft=draft))
        draft = response

        if "[FINISH]" in response:
            trace.append(step("reflect", step_no=i + 1, verdict="FINISH"))
            trace.append(step("finalize", reason="self_finish"))
            return _strip_self_rag_markers(response), trace

        trace.append(step("reflect", step_no=i + 1, verdict="CONTINUE"))

    trace.append(step("finalize", reason="max_steps_reached"))
    return _strip_self_rag_markers(draft), trace


def inspect_self_rag(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    answer, trace = self_rag_pipeline(question)
    print(f"🧠 最终答案：\n{answer}\n")
    print("🔍 流程 trace：")
    print_trace(trace)


inspect_self_rag(demo_question("self_rag"))


### 看 trace 学到了什么

```
[1] retrieve  step_no=1, n_hits=4
[2] reflect   step_no=1, verdict='CONTINUE'   ← 模型自评：还差点
[3] retrieve  step_no=2, n_hits=4              ← 自己又来了一轮
[4] reflect   step_no=2, verdict='FINISH'
[5] finalize  reason='self_finish'
```

注意它和迭代检索的 trace 形状几乎一样——都是"反思 → 决定"。
关键差别在于**反思的对象**：迭代是反思"信息够不够"，Self-RAG 是反思"答案够不够"。
在原版论文里，这些反思都由 reflection token 在模型推理时**一次性**输出，
不需要像本节这样多调一次 LLM。
本次运行中 Self-RAG 代表题一轮即 `[FINISH]`。这正好暴露教学化简版的局限：通用 LLM 的自评往往偏自信，不能稳定复现论文原版 reflection token 的继续检索行为。


## 横向对比与选型

到这里 6 种方法都过了一遍。下面这张图把它们的"决策插入位置"放在同一根流水线上：

```
                                            生成之前      生成之中      生成之后
┌─────────────────┐ ┌────────────┐ ┌────────────┐ ┌────────────┐ ┌──────────┐
│  question       │→│   route?   │→│  retrieve  │→│  generate  │→│ reflect? │→ answer
└─────────────────┘ └────────────┘ └────────────┘ └────────────┘ └──────────┘
                          │              │                              │
                          │              │                              │
                     ┌────┴────┐    ┌────┴────┐                    ┌────┴────┐
                     │  路由    │    │  CRAG   │                    │ 迭代    │
                     │  自适应  │    │ (挑证据) │                    │ Self-RAG│
                     │  递归    │    └─────────┘                    └─────────┘
                     │ (拆题)   │
                     └─────────┘
```

### 选型决策树

```
你的问题主要是哪种"痛"？
├─ 单轮回答总是漏点东西        → 迭代检索 / Self-RAG
├─ 问题明显是 N 件事的合体     → 递归检索
├─ 知识库分了多个索引/工具     → 查询路由
├─ 用户提问难度跨度大         → 自适应检索（外层套路由）
├─ 检索结果噪声大            → Corrective RAG
└─ 多种痛同时存在            → 组合：路由 → 自适应 → CRAG → 生成
```

> **一个常见误区**：上一个方法不是必须升级到下一个方法的关系。
> 真实业务里，先把单轮 baseline 跑稳，再**针对你测出来的具体痛点**挑一个方法上，
> 比一上来就堆 6 种方法效果更好（也更便宜）。

## 跑一次完整评估

下面把 baseline + 6 种流程增强方法都在同一题集上跑一次，用维度计分裁判打分。

> ⚠️ **预期耗时**：本节方法都是多步 pipeline，每道题可能调多次 LLM。
> 想快速验收，可以临时改成 `QA_INDICES = QA_INDICES_CONCEPT[:3]`。


In [ ]:
def flatten_demo_indices(demo_cases: dict[str, list[int]]) -> list[int]:
    return sorted({idx for values in demo_cases.values() for idx in values})

EVAL_INDICES = sorted(set(QA_INDICES) | set(flatten_demo_indices(DEMO_CASES)))
qna_dict = load_qna_subset(QA_PATH, EVAL_INDICES)

METHOD_RUNS = {
    "baseline": baseline_pipeline,
    "iterative": iterative_pipeline,
    "recursive": recursive_pipeline,
    "routing": routing_pipeline,
    "adaptive": adaptive_pipeline,
    "crag": crag_pipeline,
    "self_rag": self_rag_pipeline,
}

METHOD_DFS: dict[str, pd.DataFrame] = {}
METHOD_RECORDS: dict[str, list[dict]] = {}

for method_name, pipeline_fn in METHOD_RUNS.items():
    method_df, records = eval_pipeline_with_trace(method_name, pipeline_fn, qna_dict)
    METHOD_DFS[method_name] = method_df
    METHOD_RECORDS[method_name] = records

baseline_df  = METHOD_DFS["baseline"]
iterative_df = METHOD_DFS["iterative"]
recursive_df = METHOD_DFS["recursive"]
routing_df   = METHOD_DFS["routing"]
adaptive_df  = METHOD_DFS["adaptive"]
crag_df      = METHOD_DFS["crag"]
selfrag_df   = METHOD_DFS["self_rag"]


In [ ]:
compare_df = build_compare_table(
    [baseline_df, iterative_df, recursive_df, routing_df, adaptive_df, crag_df, selfrag_df],
    names=["baseline", "iterative", "recursive", "routing", "adaptive", "crag", "self_rag"],
)
compare_df


## 分方法真实差异面板

主表回答"整体趋势如何"；下面的 method case panel 回答"在方法适配题型上，机制是否真的触发，证据或答案覆盖是否改善"。两者来自同一轮 `METHOD_DFS`、`METHOD_RECORDS`、`baseline_df`、方法 `*_df` 和同一个维度计分裁判，只是展示层次不同。面板不得重新调用 pipeline 生成 trace。

面板中的 `AssertionError` 是交付阻断：如果 trace 不满足方法机制要求，该 case 不能通过验收。本节评估是教学评估，不是 SOTA 排名，也不是开放域 benchmark。样本量小，裁判是 LLM-as-judge，结果会受模型版本、限流重试、检索缓存和问题选择影响。主表用于观察整体趋势；分方法 case panel 用于观察适配题型上的机制差异。



In [ ]:
BASELINE_RECORDS_BY_QUESTION = {
    r["question"]: r for r in METHOD_RECORDS["baseline"]
}


def records_for_cases(method_name: str, case_keys: list[str]) -> list[dict]:
    expected_questions = []
    for case_key in case_keys:
        case_qna = load_qna_subset(QA_PATH, DEMO_CASES[case_key])
        expected_questions.extend((case_key, question) for question in case_qna.keys())

    available = {r["question"]: r for r in METHOD_RECORDS[method_name]}
    missing = [q for _, q in expected_questions if q not in available]
    if missing:
        raise ValueError(
            f"{method_name} targeted cases are not in qna_dict / METHOD_RECORDS: {missing[:2]}"
        )
    records = []
    for case_key, question in expected_questions:
        record = {**available[question], "case_key": case_key}
        records.append(record)
    return records


def trace_shape(trace: list[dict]) -> str:
    return " -> ".join(s.get("kind", "") for s in trace)


def first_retrieve(record: dict) -> dict:
    return next((s for s in record["trace"] if s.get("kind") == "retrieve"), {})


def display_flow_method_panel(method_label: str, method_name: str, case_keys: list[str]):
    records = records_for_cases(method_name, case_keys)
    validations = [validate_demo_trace(method_name, r["trace"], case_key=r["case_key"]) for r in records]

    print(f"### {method_label} 真实结果")
    for record, validation in zip(records, validations):
        base = BASELINE_RECORDS_BY_QUESTION[record["question"]]
        score_delta = record["score"] - base["score"]
        coverage_deltas = [
            s.get("answer_coverage_delta")
            for s in record["trace"]
            if isinstance(s.get("answer_coverage_delta"), (int, float))
        ]
        relation = (
            "win" if record["score"] > base["score"]
            else "regression" if record["score"] < base["score"]
            else "tie"
        )
        print(f"- case_key: {record['case_key']}")
        print(f"  分数关系: {relation} ({record['score']} vs baseline {base['score']}; delta={score_delta})")
        print(f"  问题: {record['question']}")
        print(f"  Baseline answer: {short_text(base['answer'], 120)}")
        print(f"  Baseline trace: {trace_shape(base['trace'])}")
        print(f"  Method answer: {short_text(record['answer'], 120)}")
        print(f"  Method trace: {validation['kinds']}")
        if coverage_deltas:
            print(f"  Coverage delta: {coverage_deltas}")
        print(f"  Trace 检查: {'通过' if validation['ok'] else '需解读'} - {validation['expected']}")
        print_trace(record["trace"])

    failed = [v for v in validations if not v["ok"]]
    if failed:
        raise AssertionError(f"{method_name} trace validation failed: {failed}")

    if method_name == "routing":
        expected_by_key = {
            "routing_general": "general_index",
            "routing_math": "math_index",
        }
        retrieve_by_key = {r["case_key"]: first_retrieve(r) for r in records}
        route_by_key = {
            r["case_key"]: next((s for s in r["trace"] if s.get("kind") == "route"), {})
            for r in records
        }
        for case_key, expected_branch in expected_by_key.items():
            actual_branch = retrieve_by_key[case_key].get("branch")
            low_confidence_mixed = (
                actual_branch == "mixed_index"
                and route_by_key[case_key].get("confidence", 1.0) < 0.55
            )
            assert actual_branch == expected_branch or low_confidence_mixed, retrieve_by_key
        shapes = {
            (
                s.get("branch"),
                s.get("n_hits"),
                s.get("context_chars"),
                s.get("first_hit"),
                s.get("selected_hits"),
            )
            for s in retrieve_by_key.values()
        }
        assert len(shapes) >= 2, "routing branch traces must differ in branch/n_hits/context_chars/first_hit/selected_hits"
    if method_name == "adaptive":
        delegate_by_key = {
            r["case_key"]: [
                s.get("target") or s.get("delegate")
                for s in r["trace"]
                if s.get("kind") == "delegate"
            ]
            for r in records
        }
        assert all(delegate_by_key.get(k) for k in ["adaptive_simple", "adaptive_moderate", "adaptive_complex"]), delegate_by_key
        delegates = {d for values in delegate_by_key.values() for d in values if d}
        assert len(delegates) >= 2, delegates
    if method_name == "self_rag":
        formal_benefit = any(
            r["score"] >= BASELINE_RECORDS_BY_QUESTION[r["question"]]["score"]
            or any(
                isinstance(s.get("answer_coverage_delta"), (int, float))
                and s.get("answer_coverage_delta") >= 0
                for s in r["trace"]
            )
            for r in records
        )
        assert formal_benefit, "self_rag needs at least one formal case with non-negative coverage delta or baseline benefit"


display(flow_score_summary(compare_df))

display_flow_method_panel("Iterative", "iterative", ["iterative"])
display_flow_method_panel("Recursive", "recursive", ["recursive"])
display_flow_method_panel("Routing", "routing", ["routing_general", "routing_math"])
display_flow_method_panel("Adaptive", "adaptive", ["adaptive_simple", "adaptive_moderate", "adaptive_complex"])
display_flow_method_panel("CRAG", "crag", ["crag"])
display_flow_method_panel("Self-RAG", "self_rag", ["self_rag"])


## 怎么读这张对比表

`compare_df`：**每行一题、每列一个方法**，单元格是维度计分裁判给的 0/1/2 分。

但**直接看分数高低意义有限**，因为：
- 路由的天花板取决于"是否真的有多个索引"——本节是模拟，看不出收益
- 递归在"可拆题"上得分高，在"原子题"上反而可能因为过度拆解而扣分
- Self-RAG 的方差大，单次实验不一定稳定

更好的读法是**结合 trace**：
1. 找一道方法 A 比方法 B 高分的题
2. 把两个方法的 trace 都打印出来对比
3. 看流程在哪一步真正走出了不同——这才是"流程增强"的教学价值

## 学习检查点

读完本节，你应该能回答：

- [ ] **接口约定**：本节所有 `*_pipeline` 的输入输出是什么？为什么要返回 `trace`？
- [ ] **三类决策点**：路由、CRAG、Self-RAG 分别把决策插在了 RAG 流水线的哪一步？
- [ ] **迭代 vs 递归**：两者都在"分子问题"，本质区别是什么？
- [ ] **路由 vs 自适应**：横向选择 vs 纵向选择是什么意思？
- [ ] **Self-RAG 教学版的简化**：原版用什么机制做"是否继续"的判断？教学版又是怎么做的？
- [ ] **何时升级到下一章**：当流程增强还不够时，会出现什么样的需求？


如果某个方法在主表中没有提分，不要直接写成"方法无效"。先回到该方法的 trace 定位失败环节：router 是否选错分支、grader 是否过滤了有用证据、recursive 是否拆出重复子问题、Self-RAG 是否过早 `FINISH`、生成阶段是否没有吸收新增证据。只有 trace 能说明失败发生在哪里。


## 实测结果与解读

下面的汇总来自当前 notebook 的 `compare_df`，因此会随模型输出、限流重试、题集调整和随机波动略有变化。读这张表时不要只看均值，更要结合上面的分方法 trace：流程增强的教学重点是“哪一步做了不同决策”。


In [ ]:
summary_df = flow_score_summary(compare_df)
display(summary_df)

baseline_mean = float(compare_df["baseline"].mean())
rows = []
for method in ["iterative", "recursive", "routing", "adaptive", "crag", "self_rag"]:
    method_mean = float(compare_df[method].mean())
    rows.append({
        "方法": method,
        "均值": round(method_mean, 3),
        "Δ vs baseline": round(method_mean - baseline_mean, 3),
        "严格提分题数": int((compare_df[method] > compare_df["baseline"]).sum()),
        "回退题数": int((compare_df[method] < compare_df["baseline"]).sum()),
    })

display(pd.DataFrame(rows))


### 如何解读本轮结果

1. 如果 `iterative` 提分，优先查看它的 trace 是否真的发生了第二轮检索；如果没有第二轮却提分，说明收益可能来自生成随机性，而不是补检索。
2. 如果 `recursive` 在复杂题上提分，查看 `decompose` 是否拆出了互补子问题；如果在简单题回退，通常是过度拆解造成的。
3. 如果 `routing` 与 baseline 接近，先看两个 branch 的 `first_hit` 和 `n_hits` 是否不同；如果 branch 不同但得分相同，说明路由改变了上下文但未改变裁判分数。
4. 如果 `crag` 提分，查看被过滤的 `partial/irrelevant` 证据；如果回退，可能是 grader 把有用片段降权或过滤了。
5. 如果 `self_rag` 很少触发 `CONTINUE`，这是教学化简版的正常局限：通用 LLM 往往过早自信，不像论文原版有专门 reflection token。


## 与下一节（系统增强）的衔接

本节所有方法都假设了一件事：**系统是单一智能体、单次会话、无记忆**。
但实际产品里常见这些"流程也救不了"的需求：

- 多个文档智能体要**协作**（一个查论文、一个查代码、再有一个汇总）
- 需要**跨会话**记住用户偏好（"上次他问 SVM 时偏好数学推导"）
- 同一问题要在**多轮对话**里持续推进，而不是每次都重新开始

这些就不是"流程"层能解决的，而要把系统**升一个抽象层**——
进入下一节：`3. 系统增强.ipynb`。